# Notebook 03 — SQL Quizzes Practice Lab (23 Exercises)

**Session:** Database - SQL (1)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Execute and verify all 23 single-table SQL quiz solutions against `sql_store`. Where a quiz uses MySQL-specific scalar functions (`YEAR`, `MONTH`, `DAYOFWEEK`, `LOG2`, `FLOOR`, `REGEXP`), we register lightweight SQLite UDF equivalents so the SQL logic runs identically in local Python while highlighting dialect differences.

In [ ]:
import datetime as dt
import math
import re
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))

# Register MySQL-compatible helper functions for portable local testing
conn.create_function(
    "REGEXP",
    2,
    lambda pat, val: bool(re.search(pat, val, re.IGNORECASE)) if val is not None else False,
)
conn.create_function("YEAR", 1, lambda d: int(d[:4]) if d else None)
conn.create_function("MONTH", 1, lambda d: int(d[5:7]) if d else None)
conn.create_function("FLOOR", 1, lambda x: math.floor(x) if x is not None else None)
conn.create_function("LOG2", 1, lambda x: math.log2(x) if x is not None and x > 0 else None)


def mysql_dayofweek(d: str | None) -> int | None:
    if not d:
        return None
    # Python weekday(): Monday=0..Sunday=6 -> MySQL DAYOFWEEK(): Sunday=1..Saturday=7
    py_wd = dt.date.fromisoformat(d).weekday()
    return ((py_wd + 1) % 7) + 1


conn.create_function("DAYOFWEEK", 1, mysql_dayofweek)


def query(sql: str) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = conn.execute(sql)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))

## Quizzes 1–8: Basic Filtering, Sorting, and Date Predicates

In [ ]:
q1 = query("SELECT first_name, last_name FROM customers WHERE points > 1000;")
q2 = query("SELECT name, unit_price FROM products WHERE quantity_in_stock BETWEEN 50 AND 100;")
q3 = query(
    "SELECT order_id, order_date FROM orders WHERE order_date BETWEEN '2018-01-01' AND '2018-12-31';"
)
q4 = query("SELECT first_name, last_name FROM customers WHERE phone IS NULL;")
q5 = query("SELECT customer_id, points FROM customers WHERE first_name LIKE 'I%';")
q6 = query("SELECT order_id, order_date FROM orders ORDER BY order_date DESC LIMIT 5;")
q7 = query("SELECT name, unit_price FROM products ORDER BY unit_price DESC LIMIT 3;")
q8 = query(
    """
    SELECT first_name, last_name
    FROM customers
    WHERE YEAR(birth_date) % 4 = 0
      AND (YEAR(birth_date) % 100 != 0 OR YEAR(birth_date) % 400 = 0);
    """
)

print("Q1 (>1000 pts):", len(q1))
print("Q2 (stock 50..100):", len(q2))
print("Q3 (2018 orders):", len(q3))
print("Q4 (NULL phone):", [r["first_name"] for r in q4])
print("Q5 (first_name I%):", [r["customer_id"] for r in q5])
print("Q6 (latest 5 orders):", [r["order_id"] for r in q6])
print("Q7 (top 3 expensive):", [r["name"] for r in q7])
print("Q8 (leap year birth):", [r["first_name"] for r in q8])
assert len(q6) == 5 and len(q7) == 3
assert "Alana" in [r["first_name"] for r in q8]

## Quizzes 9–16: Grouping, Regex, Subqueries, Pagination, and Windows

In [ ]:
q9 = query(
    """
    SELECT customer_id, COUNT(order_id) AS order_count
    FROM orders
    GROUP BY customer_id
    ORDER BY order_count DESC;
    """
)
q10 = query("SELECT name, quantity_in_stock FROM products WHERE quantity_in_stock % 10 = 0;")
q11 = query("SELECT name, unit_price FROM products WHERE name REGEXP '^[aeiouAEIOU]';")
q12 = query(
    """
    SELECT customer_id, SUM(points) AS total_points
    FROM customers
    GROUP BY customer_id
    HAVING SUM(points) > (SELECT AVG(points) FROM customers);
    """
)
# Portable SQLite/MySQL syntax for MySQL's `LIMIT 4, 6` is `LIMIT 6 OFFSET 4`
q13 = query("SELECT order_id, order_date FROM orders ORDER BY order_date LIMIT 6 OFFSET 4;")
q14 = query(
    """
    SELECT customer_id, SUM(points) OVER (PARTITION BY MONTH(birth_date)) AS total_points
    FROM customers
    ORDER BY total_points DESC;
    """
)
q15 = query("SELECT name, unit_price FROM products WHERE unit_price = FLOOR(unit_price);")
q16 = query(
    """
    SELECT customer_id, COUNT(order_id) AS order_count
    FROM orders
    GROUP BY customer_id
    HAVING order_count > 3;
    """
)

print("Q9 top customer:", q9[0])
print("Q10 multiple of 10:", [r["name"] for r in q10])
print("Q11 starts with vowel:", [r["name"] for r in q11])
print("Q12 above avg points count:", len(q12))
print("Q13 5th-10th orders:", [r["order_id"] for r in q13])
print("Q14 window birth month top 3:", q14[:3])
print("Q15 whole unit_price:", [r["name"] for r in q15])
print("Q16 >3 orders:", q16)
assert len(q13) == 6
assert len(q16) >= 1

## Quizzes 17–23: Correlated Subqueries, Date Arithmetic, Powers of 2, and Weekends

In [ ]:
q17 = query(
    """
    SELECT name, unit_price
    FROM products
    WHERE unit_price > (SELECT AVG(unit_price) FROM products);
    """
)
q18 = query(
    """
    SELECT o.order_id, o.order_date
    FROM orders o
    WHERE o.order_date = (
        SELECT MIN(order_date)
        FROM orders
        WHERE customer_id = o.customer_id
    );
    """
)
# In MySQL: WHERE order_date >= CURDATE() - INTERVAL 7 DAY
# In SQLite (referenced against 2026-11-14): DATE('2026-11-14', '-7 days')
q19 = query(
    """
    SELECT order_id, order_date
    FROM orders
    WHERE order_date >= DATE('2026-11-14', '-7 days');
    """
)
q20 = query(
    """
    SELECT customer_id, points
    FROM customers
    WHERE first_name LIKE '%a%' AND last_name LIKE '%b%';
    """
)
q21 = query(
    """
    SELECT customer_id, COUNT(order_id) AS order_count
    FROM orders
    GROUP BY customer_id
    ORDER BY order_count;
    """
)
q22 = query(
    """
    SELECT name, quantity_in_stock
    FROM products
    WHERE quantity_in_stock > 0
      AND LOG2(quantity_in_stock) = FLOOR(LOG2(quantity_in_stock));
    """
)
q23 = query(
    """
    SELECT order_id, order_date
    FROM orders
    WHERE DAYOFWEEK(order_date) IN (1, 7);
    """
)

print("Q17 above avg price:", len(q17))
print("Q18 oldest order per customer:", len(q18))
print("Q19 orders in last 7 days:", [r["order_id"] for r in q19])
print("Q20 first_name '%a%' & last_name '%b%':", [r["customer_id"] for r in q20])
print("Q21 ascending order count top 3:", q21[:3])
print("Q22 power-of-2 stock:", q22)
print("Q23 weekend orders:", [r["order_id"] for r in q23])
assert len(q19) >= 1
assert len(q22) >= 2
assert len(q23) >= 1